# Lezione 9 — Caso práctico: define tu objetivo con dati de criptomonedas

**Manuale associato:** [manual/09_caso_cripto_define_objetivo.md](../manual/09_caso_cripto_define_objetivo.md)

## Resumen para la lezione

- El objetivo es clasificación binaria (sube/baja), no predicción exacta del precio.
- Combinar varias criptomonedas correlacionadas da más contexto que mirar solo una.
- `yfinance` descarga históricos reales sin depender de archivos externos.
- La columna "target" se construye comparando el precio actual contra un precio futuro desplazado.

In [ ]:
!pip install --quiet yfinance

import yfinance as yf
import pandas as pd

tickers = ["BTC-USD", "LTC-USD", "BCH-USD", "ETH-USD"]
datos_por_moneda = {}

for ticker in tickers:
    df = yf.download(ticker, period="2y", interval="1d")
    df = df[["Close", "Volume"]].rename(columns={"Close": "close", "Volume": "volume"})
    datos_por_moneda[ticker] = df

datos_por_moneda["LTC-USD"].head()

In [ ]:
main_df = pd.DataFrame()

for ticker, df in datos_por_moneda.items():
    prefijo = ticker.split("-")[0]
    df = df.rename(columns={"close": f"{prefijo}_close", "volume": f"{prefijo}_volume"})
    main_df = df if main_df.empty else main_df.join(df, how="outer")

main_df = main_df.ffill()
main_df.dropna(inplace=True)
main_df.head()

In [ ]:
RATIO_A_PREDECIR = "LTC"
PERIODO_FUTURO = 3

main_df["future"] = main_df[f"{RATIO_A_PREDECIR}_close"].shift(-PERIODO_FUTURO)


def clasifica(precio_actual, precio_futuro):
    return 1 if float(precio_futuro) > float(precio_actual) else 0


main_df["target"] = list(map(clasifica, main_df[f"{RATIO_A_PREDECIR}_close"], main_df["future"]))
main_df.dropna(inplace=True)

main_df[[f"{RATIO_A_PREDECIR}_close", "future", "target"]].head(10)

In [ ]:
print(main_df["target"].value_counts(normalize=True))

## Cierre de la lezione

- Guarda `main_df` (por ejemplo, con `main_df.to_pickle("main_df.pickle")`): la lezione 10 continúa desde aquí.
- Anota el balance de leziones que viste arriba: lo vas a necesitar para la lezione 11.

In [ ]:
main_df.to_pickle("main_df.pickle")